# 06 · Econometría + Champion/Challenger
Un producto maduro no elige ML por moda. Compara: **baseline económico**, **modelo interpretable**, **modelo no lineal** y **modelo de conteos**.

La econometría aquí sirve para estructura, inferencia condicionada y diagnóstico; los modelos ML sirven para error predictivo. Ninguno se declara causal sin identificación.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'replica_cygnus').exists())
sys.path.insert(0, str(ROOT/'src'))
from replica_cygnus.economic_intelligence import install_feature_mart, load_monthly_panel, build_supervised_panel
from replica_cygnus.economic_intelligence.features import select_model_matrix
from replica_cygnus.economic_intelligence.models import backtest_regressors
install_feature_mart(); panel = load_monthly_panel(); sup = build_supervised_panel(panel)


## Panel econométrico: absorción neta
Controlamos por proyecto (efectos fijos) y estacionalidad. Esto ayuda a distinguir diferencias estructurales entre proyectos de variación dentro del tiempo.

In [ ]:
reg = sup[['proyecto','absorcion_neta_mes','absorcion_lag1','stock_inicio_observado','edad_comercial_meses','absorcion_neta_market','season_sin','season_cos']].replace([np.inf,-np.inf],np.nan).dropna().copy()
formula = 'absorcion_neta_mes ~ absorcion_lag1 + np.log1p(stock_inicio_observado) + edad_comercial_meses + absorcion_neta_market + season_sin + season_cos + C(proyecto)'
ols = smf.ols(formula, data=reg).fit(cov_type='HC3')
print(ols.summary())

## Poisson para intensidad de minutas
Minutas son conteos. Usamos stock como exposición mediante `log(stock+1)`. El resultado es intensidad condicional, no causalidad.

In [ ]:
pois = sup[['proyecto','ventas_minutas_mes','minutas_lag1','stock_inicio_observado','edad_comercial_meses','season_sin','season_cos']].replace([np.inf,-np.inf],np.nan).dropna().copy()
pois['log_exposure'] = np.log1p(pois['stock_inicio_observado'].clip(lower=0))
glm = smf.glm('ventas_minutas_mes ~ minutas_lag1 + edad_comercial_meses + season_sin + season_cos + C(proyecto)', data=pois, family=sm.families.Poisson(), offset=pois['log_exposure']).fit()
print(glm.summary())

## Challenger predictivo
Ridge = lineal regularizado; Gradient Boosting = no lineal; Random Forest = interacciones; Poisson = conteos cuando aplica.

In [ ]:
X, y, cols = select_model_matrix(sup, 'target_mov_neto_next_1m', include_macro=True, include_reference=False)
meta = sup.loc[X.index, ['periodo_mes','codigo_proyecto','proyecto']]
scores_net, _, preds_net = backtest_regressors(X, y, meta, test_months=6)
scores_net

In [ ]:
Xm, ym, _ = select_model_matrix(sup, 'target_minutas_next_1m', include_macro=True, include_reference=False)
meta_m = sup.loc[Xm.index, ['periodo_mes','codigo_proyecto','proyecto']]
scores_min, _, preds_min = backtest_regressors(Xm, ym, meta_m, nonnegative_target=True, test_months=6)
scores_min

## Gate de promoción
Un champion sólo se promueve si: 1) supera baseline en holdout temporal; 2) el error es estable por proyecto; 3) no usa features con leakage; 4) pasa sanity checks económicos; 5) documenta horizonte, universo y fecha de entrenamiento.